# OEC ACE-Step 1.5 — Gradio client for the isolated music model server

**UI environment:** `phase2_3d`; this notebook does not import the ACE-Step ML stack.  
**Backend:** `~/.local/venvs/oec-acestep-py311/bin/python` on a separately managed service at `127.0.0.1:7870`.

From the OEC repository root, after checking A100 ownership and StorageManager status:

```bash
python tools/oec_service_manager.py start audio-api --provider ace-step --gpu <ID> --allow-download
python tools/oec_service_manager.py status audio-api
python tools/oec_service_manager.py logs audio-api --tail 80
```

The backend must report `ready=true` and `model_loaded=true`. `--allow-download` is optional and must only be used when missing checkpoints may be downloaded through fresh StorageManager admission. Never kill an unrelated GPU process or bypass quota guards. The API is loopback-only; SSH-tunnel it when accessing the page remotely.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import time
import urllib.parse
import gradio as gr
import requests
import soundfile as sf

API_BASE = os.environ.get("OEC_MEDIA_API_URL", "http://127.0.0.1:7870").rstrip("/")
RESULT_ROOT = Path("/tmp") / "oec-notebook-results"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
if urllib.parse.urlsplit(API_BASE).hostname not in {"127.0.0.1", "localhost", "::1"}:
    raise RuntimeError("ACE-Step notebook only permits a loopback OEC media API endpoint.")
HTTP = requests.Session()
HTTP.trust_env = False

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def service_health():
    try:
        response = HTTP.get(f"{API_BASE}/health", timeout=8)
        payload = response.json()
    except Exception as exc:
        raise gr.Error(
            f"ACE-Step model server is unreachable at {API_BASE}. Start the isolated ace-step service first."
        ) from exc
    if response.status_code != 200 or not payload.get("ready") or not payload.get("model_loaded"):
        raise gr.Error(f"ACE-Step is not ready: {json.dumps(payload, ensure_ascii=False)}")
    return payload

def submit_and_wait(payload, timeout_seconds=1800):
    health = service_health()
    response = HTTP.post(f"{API_BASE}/v1/jobs", json=payload, timeout=30)
    if response.status_code != 202:
        raise gr.Error(f"ACE-Step job submission refused (HTTP {response.status_code}): {response.text[:1200]}")
    accepted = response.json()
    job_id = accepted.get("job_id")
    if not isinstance(job_id, str) or not job_id.startswith("job_"):
        raise gr.Error(f"Unexpected ACE-Step job response: {accepted}")
    deadline = time.monotonic() + timeout_seconds
    latest = accepted
    while time.monotonic() < deadline:
        poll = HTTP.get(f"{API_BASE}/v1/jobs/{job_id}", timeout=20)
        if poll.status_code != 200:
            raise gr.Error(f"ACE-Step job status lookup failed (HTTP {poll.status_code}): {poll.text[:800]}")
        latest = poll.json()
        status = latest.get("status")
        if status == "succeeded":
            break
        if status in {"failed", "error", "cancelled"}:
            raise gr.Error(f"ACE-Step job {status}: {latest.get('message', 'No error details returned')}")
        time.sleep(2)
    else:
        raise gr.Error("ACE-Step job timed out; inspect the service-manager logs.")
    result = HTTP.get(f"{API_BASE}/v1/jobs/{job_id}/result", timeout=120)
    if result.status_code != 200:
        raise gr.Error(f"ACE-Step result download failed (HTTP {result.status_code}): {result.text[:800]}")
    content_type = result.headers.get("Content-Type", "").split(";", 1)[0].lower()
    ext = {"audio/flac": ".flac", "audio/wav": ".wav"}.get(content_type)
    if ext is None:
        raise gr.Error(f"Unexpected ACE-Step result content type: {content_type!r}")
    output_path = RESULT_ROOT / f"{job_id}{ext}"
    if output_path.exists():
        raise gr.Error("Refusing to overwrite an existing notebook result.")
    output_path.write_bytes(result.content)
    if output_path.stat().st_size < 64:
        output_path.unlink(missing_ok=True)
        raise gr.Error("ACE-Step returned an empty audio asset.")
    if ext == ".flac" and not output_path.read_bytes().startswith(b"fLaC"):
        output_path.unlink(missing_ok=True)
        raise gr.Error("ACE-Step returned data without a FLAC signature.")
    try:
        audio_info = sf.info(str(output_path))
    except Exception as exc:
        output_path.unlink(missing_ok=True)
        raise gr.Error("ACE-Step returned an audio file that could not be decoded.") from exc
    if audio_info.frames <= 0 or audio_info.samplerate <= 0 or audio_info.channels <= 0:
        output_path.unlink(missing_ok=True)
        raise gr.Error("ACE-Step returned an audio asset with invalid stream metadata.")
    details = {
        "job_id": job_id, "status": latest.get("status"), "api": API_BASE,
        "health": health, "metadata": latest.get("metadata", {}),
        "artifact_path": str(output_path), "artifact_bytes": output_path.stat().st_size,
        "artifact_sha256": sha256_bytes(output_path.read_bytes()),
        "sample_rate_hz": audio_info.samplerate, "channels": audio_info.channels,
        "duration_seconds": audio_info.duration, "format": audio_info.format, "subtype": audio_info.subtype,
    }
    return str(output_path), json.dumps(details, indent=2, ensure_ascii=False)

In [ ]:
def generate_music(prompt, lyrics, instrumental, duration, seed, steps, bpm, language):
    if not prompt or not prompt.strip():
        raise gr.Error("Enter a music description.")
    parameters = {
        "duration_seconds": float(duration), "seed": int(seed), "steps": int(steps),
        "lyrics": lyrics or "", "instrumental": bool(instrumental),
        "bpm": int(bpm) if bpm is not None else None, "language": language or "unknown",
    }
    return submit_and_wait({"prompt": prompt.strip(), "parameters": parameters}, timeout_seconds=1800)

demo = gr.Interface(
    fn=generate_music,
    inputs=[
        gr.Textbox(label="Music description", lines=3, value="Dreamy cinematic ambient score"),
        gr.Textbox(label="Lyrics (optional)", lines=4),
        gr.Checkbox(value=True, label="Instrumental"),
        gr.Slider(10, 180, value=10, step=1, label="Duration (seconds)"),
        gr.Number(value=-1, precision=0, label="Seed (-1 = random)"),
        gr.Slider(1, 20, value=8, step=1, label="Inference steps"),
        gr.Number(value=None, precision=0, label="BPM (optional)"),
        gr.Dropdown(["unknown", "en", "hi", "ja", "zh", "es", "fr", "de"], value="unknown", label="Vocal language"),
    ],
    outputs=[
        gr.Audio(type="filepath", label="Generated audio file"),
        gr.Textbox(label="Job metadata", lines=12),
    ],
    title="ACE-Step 1.5 — Gradio model server client",
    description="The backend runs in its isolated environment. This notebook submits jobs and decodes the resulting audio file.",
)
if os.environ.get("OEC_GRADIO_HEADLESS_SMOKE") != "1":
    demo.queue(max_size=1).launch(server_name="127.0.0.1", server_port=7903, share=False, inline=False, show_error=True)